In [15]:
import pandas as pd
import mysql.connector
import uuid
import numpy as np
import datetime

In [16]:
db_config = {
    "host": "localhost",
    "user": "your_user",
    "password": "your_password",
    "database": "your_bd",
    "port": "your port"
}

In [17]:
COST_COLUMNS = [
    "llantas", "hojalateria", "tapiceria", "cielo", "volante", 
    "funda", "mecanico", "tapones", "multa", "kit", "sombrerera", 
    "rin_llaveta", "aceite", "tapon_facion", "alineacion", "manija", "faros_calaveras"
]

In [18]:
def arreglar_fecha_excel(valor):
    if pd.isna(valor) or str(valor).strip() == '':
        return None
    val_str = str(valor).strip()
    try:
        serial = float(val_str)
        fecha = datetime.datetime(1899, 12, 30) + datetime.timedelta(days=serial)
        return fecha.strftime('%Y-%m-%d')
    except ValueError:
        return val_str

In [19]:
def procesar_csv_a_mysql():
    conn = None
    cursor = None
    try:
        conn = mysql.connector.connect(**db_config)
        cursor = conn.cursor()
        print("✅ Conectado a MySQL exitosamente.")

        print("Lectura del archivo bazar.csv...")
        df = pd.read_csv('bazar.csv')
        df = df.replace({np.nan: None}) 

        autos_ok = 0
        costos_ok = 0

        for index, row in df.iterrows():
            car_id = str(uuid.uuid4()) 
            
            brand = str(row.get('brand', '')).strip()[:20] 
            model = str(row.get('model', '')).strip()[:50]
            color = str(row.get('color', '')).strip()[:15]
            
            try:
                year_of_car = int(float(row['year_of_car'])) if row['year_of_car'] is not None else 2000
            except:
                year_of_car = 2000
                
            purchase_date = arreglar_fecha_excel(row.get('purchase_date'))
            if purchase_date is None:
                purchase_date = '2024-01-01'

            sale_date = arreglar_fecha_excel(row.get('sale_date'))

            csv_created = row.get('created')
            if pd.notna(csv_created) and str(csv_created).strip() != '':
                created_raw = str(csv_created).strip()
                if created_raw.endswith('.0'):
                    created_raw = created_raw[:-2]
                    
                if len(created_raw) == 4:
                    created = f"{created_raw}-01-01 00:00:00"
                else:
                    created = created_raw
            else:
                created = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')

            try:
                purchase_price = float(row['purchase_price']) if row['purchase_price'] is not None else 0.0
            except:
                purchase_price = 0.0
                
            try:
                price_for_sale = float(row['price_for_sale']) if row['price_for_sale'] is not None else 0.0
            except:
                price_for_sale = 0.0

            status = 'Vendido' if sale_date is not None else 'Inventario'

            query_car = """
                INSERT INTO cars 
                (car_id, brand, model, year_of_car, color, purchase_date, sale_date, status, purchase_price, price_for_sale, created)
                VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
            """
            val_car = (car_id, brand, model, year_of_car, color, purchase_date, sale_date, status, purchase_price, price_for_sale, created)

            try:
                cursor.execute(query_car, val_car)
                autos_ok += 1
                
                for cost_type in COST_COLUMNS:
                    if cost_type in df.columns:
                        val_cost = row[cost_type]
                        if val_cost is not None and str(val_cost).strip() != '':
                            try:
                                amount = float(val_cost)
                                if amount > 0:
                                    cost_id = str(uuid.uuid4()) 
                                    query_cost = """
                                        INSERT INTO car_costs (cost_id, car_id, cost_type, amount, note)
                                        VALUES (%s, %s, %s, %s, %s)
                                    """
                                    cursor.execute(query_cost, (cost_id, car_id, cost_type, amount, ""))
                                    costos_ok += 1
                            except ValueError:
                                pass
            except Exception as e:
                print(f"⚠️ Error en Fila {index+2} | Auto: {brand} {model}")
                print(f"   Detalle del error MySQL: {e}")
                print("-" * 40)

        conn.commit()
        print("-" * 30)
        print("🎉 IMPORTACIÓN FINALIZADA")
        print(f"🚗 Autos registrados: {autos_ok}")
        print(f"💰 Costos registrados: {costos_ok}")

    except Exception as general_error:
        print(f"❌ Error crítico en la ejecución: {general_error}")
        if conn and conn.is_connected():
            conn.rollback() # Si algo truena, liberamos el bloqueo de la tabla
            print("🔄 Transacción revertida para evitar bloqueos.")
            
    finally:
        # Esto SIEMPRE se ejecuta al final, pase lo que pase, para limpiar todo
        if cursor is not None:
            cursor.close()
        if conn is not None and conn.is_connected():
            conn.close()
            print("🔌 Conexión a MySQL cerrada correctamente.")

procesar_csv_a_mysql()

✅ Conectado a MySQL exitosamente.
Lectura del archivo bazar.csv...
------------------------------
🎉 IMPORTACIÓN FINALIZADA
🚗 Autos registrados: 85
💰 Costos registrados: 211
🔌 Conexión a MySQL cerrada correctamente.
